In [1]:
import sys
from pathlib import Path

def add_repo_to_path():
    repo_root = Path.cwd().parent.parent
    sys.path.insert(0, str(repo_root))

In [2]:
import numpy as np
import matplotlib.pyplot as plt

from qdesignoptimizer.sim_plot_progress import DataExtractor, OptimizationPlotter
from qdesignoptimizer.sim_plot_progress import OptPltSet
from qdesignoptimizer.utils.names_parameters import (
    param,
    param_capacitance,
    param_nonlin,
)

from utils_plot import get_color_shades, crop_image_by_fraction

In [3]:
add_repo_to_path()
import tutorials.examples_coupled_transmon_chip.names as n
repo_root = Path.cwd().parent.parent
sys.path.insert(0, str(repo_root)+'/tutorials/examples_coupled_transmon_chip')
import tutorials.examples_coupled_transmon_chip.optimization_targets as ot

# Figures 

## colors

In [4]:
# selected colors based on Gothenburg tram line map
colors_got = {'green': '#009639', 
              'blue': '#0066CC', 
              'yellow': '#FFD200', 
              'red': '#E30613', 
              'pink': '#EC008C', 
              'lightblue': '#00B5E2', 
              'purple': '#93328E', 
              'orange': '#F58220', 
              'grey': '#8C8C8C',
              'lightsalmon': '#FF9E80'}

# make a list of colors
color_list = list(colors_got.values())

color_shades_purple = get_color_shades(colors_got['purple'], num_shades=5)
color_shades_red = get_color_shades(colors_got['red'], num_shades=5)
color_shades_blue = get_color_shades(colors_got['blue'], num_shades=5)
color_shades_lightblue = get_color_shades(colors_got['lightblue'], num_shades=4)
color_shades_grey = get_color_shades(colors_got['grey'], num_shades=5)

## Fig S1: Kappa vs nbr of mesh refinement passes

In [5]:
# load data
data_path = Path(r'..\data\simulations_single_qubit\simulation_single_qubit_resonator_kappa_vs_passes.txt')

def _parse_sections(filepath):
    sections, current = [], []
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                if current:
                    sections.append(np.array(current))
                    current = []
                continue
            current.append([float(x) for x in line.split(',')])
    if current:
        sections.append(np.array(current))
    return sections

freq_raw, kappa_raw, time_raw = _parse_sections(data_path)
passes     = freq_raw[:, 0].astype(int)   # shape (8,)
freq_vals  = freq_raw[:, 1:]              # shape (8, 5), GHz
kappa_vals = kappa_raw[:, 1:] / 1e3      # shape (8, 5), kHz
time_vals  = time_raw[:, 1:]             # shape (8, 5), s

In [ ]:
figure_width_inch = 3.35
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'font.size': 10,
    'mathtext.fontset': 'stix',
})

fig = plt.figure(figsize=(figure_width_inch, 3.5))
gs = fig.add_gridspec(2, 2,)
ax_a = fig.add_subplot(gs[0, :])
ax_b = fig.add_subplot(gs[1, 0])
ax_c = fig.add_subplot(gs[1, 1])

col_kappa = colors_got['blue']
col_freq  = colors_got['green']
col_time  = colors_got['orange']

for ax, vals, col, ylabel, label in [
    (ax_a, kappa_vals, col_kappa, r'$\kappa/2\pi$ (kHz)', '(a)'),
    (ax_b, freq_vals,  col_freq,  r'$f$ (GHz)',           '(b)'),
    (ax_c, time_vals,  col_time,  r'time (s)',            '(c)'),
]:
    mean = vals.mean(axis=1)
    std  = vals.std(axis=1)
    ax.errorbar(passes, mean, yerr=std,
                fmt='o-', color=col, capsize=3,
                markersize=4, linewidth=1)
    ax.set_xlabel('# passes')
    ax.set_ylabel(ylabel)
    if ax == ax_a: 
        ax.text(0.05, 0.98, label, transform=ax.transAxes,
                fontsize=10, va='top')
    else: 
        ax.text(0.1, 0.98, label, transform=ax.transAxes,
                fontsize=10, va='top')
        
fig.tight_layout()
fig.savefig(r'..\data\figures' + r"\FigS1_kappa_vs_passes.png", dpi=300)
fig.show()

## Fig S1b: Relative deviation vs nbr of mesh refinement passes

In [25]:
figure_width_inch = 3.35
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'font.size': 10,
    'mathtext.fontset': 'stix',
})

# compute relative deviation w.r.t. last-pass mean (fixed reference)
freq_mean  = freq_vals.mean(axis=1)
freq_std   = freq_vals.std(axis=1)
kappa_mean = kappa_vals.mean(axis=1)
kappa_std  = kappa_vals.std(axis=1)

freq_ref  = freq_mean[-1]   # GHz
kappa_ref = kappa_mean[-1]  # kHz

freq_rel      = (freq_mean  - freq_ref)  / freq_ref  * 100   # %
freq_rel_std  =  freq_std                / freq_ref  * 100
kappa_rel     = (kappa_mean - kappa_ref) / kappa_ref * 100
kappa_rel_std =  kappa_std               / kappa_ref * 100

fig, (ax_k, ax_f) = plt.subplots(1, 2, figsize=(figure_width_inch, 1.9))

for ax, rel, rel_std, col, ylabel, label in [
    (ax_k, kappa_rel, kappa_rel_std, colors_got['blue'],  r'$\Delta\kappa/\kappa_{10}$ (%)', '(a)'),
    (ax_f, freq_rel,  freq_rel_std,  colors_got['green'], r'$\Delta f/f_{10}$ (%)',           '(b)'),
]:
    ax.axhline(0, color='k', linewidth=0.6, linestyle='--', zorder=0)
    ax.errorbar(passes, rel, yerr=rel_std,
                fmt='o-', color=col, capsize=3,
                markersize=4, linewidth=1)
    ax.set_xlabel('passes')
    ax.set_ylabel(ylabel)
    ax.text(0.07, 0.95, label, transform=ax.transAxes,
            fontsize=10, va='top')

# annotate absolute reference value on frequency panel
ax_k.text(0.15, 0.2, f'$k_{{10}}$ = {kappa_ref:.2f} kHz',
    transform=ax_k.transAxes, fontsize=8, fontweight='regular', va='top')

ax_f.text(0.15, 0.2, f'$f_{{10}}$ = {freq_ref:.2f} GHz',
    transform=ax_f.transAxes, fontsize=8, fontweight='regular', va='top')

fig.tight_layout()
fig.savefig(r'..\data\figures' + r"\FigS1b_relative_deviation_vs_passes.png", dpi=300)
fig.show()